## PageRank Recap

**Idea:** let's assume every node in a graph has initially the same importance, and this is propagated and split among its neighbours. Repeat this propagation until convergence.

**Problems:** Does it converge?

 - Only if the original graph is **irreducible** (any state is reachable from any other state) and **aperiodic** (no “cycles” of fixed length)
 - The Web Graph does not satisfy this criteria (it has **dead ends** and **cycles**)


**Solution** (leading to the so called **Google Matrix**): 


$$\pi^{t+1} = \beta M\pi^{t} + (1-\beta) 1/n
\quad\quad \textsf{with}\quad\quad 
M[i,j] =  \begin{cases} \frac{ 1 }{o(j)} & \textsf{if}\ j\rightarrow i \\
1/n & \textsf{if}\ o(j)=0 \\
0  & \textsf{otherwise}
\end{cases}
$$

 - with probability $\beta$ we follow $M$, with probability $(1-\beta)$ we jump to a random node (**teleportation**)
 - **dead ends removal** by "linking" them to every other nodes

The PageRank $\pi$ is the stable state distribution of the above:

 - i.e., such that $\pi^{t+1}=\pi^{t}$
 - this usually happens after less than 50 iterations



# implementation 
We first import the necessary modules

In [1]:
import csv
import numpy as np
import time
from scipy import sparse
from pympler import asizeof

Our graph is represented as an adjacency matrix $n^2$ where $n$ is the number of nodes. Edges are represented by a $1$ beetween nodes.
$$A[i, j] = 1 \quad \text{if} \quad j \rightarrow i$$

In other terms, it is a binary matrix that describes if an edge exists or not.

In [ ]:
def build_adjacency_Matrix(edges, is_undirected=True):
    """
    Parameters:
    - edges: List of tuples (source, target, count)
    - is_undirected: If True, the graph is treated as undirected and the resulting matrix will be simmetric.
    Returns:
    - A: Adjacency matrix np.ndarray of shape (n, n)  
    - names: sorted list of node names, corresponding to the insexes of A (list of str) 
    """


    names = sorted({u for u, _, _ in edges} | {v for _, v, _ in edges}) # we sort names

    idx = {name: i for i, name in enumerate(names)}
    n = len(names)
    A = np.zeros((n, n), dtype=np.float64)
    for src, tgt, count in edges:
        s_id, t_id = idx[src], idx[tgt]
        A[t_id, s_id] = 1
        if is_undirected:
            A[s_id, t_id] = 1
    return A, names

Then, the actual page rank algorithm consist in a random path. More precisely, in a Markov process where at each instant, we can jump to the next node with a probability given by the matrix. The steady state represent the final computed pagerank. So, we firstly build the transition matrix.
It is a stochastic matrix: each column sums up to 1 and the values are always $\in [0,1]$, because they represents the probabilities.
In case of a dead end node, the probability is split among all the other nodes, ($1/n$). 

In [ ]:
def build_transition_matrix(A, is_sparse=False):
    """ 
    Parameters:
    - A: Adjacency matrix np.ndarray of shape (n, n)
    - is_sparse: If True, the output will be a sparse matrix (scipy.sparse.csr_matrix), otherwise a dense numpy array.
    Returns:
    - M: Transition matrix np.ndarray of shape (n, n) where M[i, j] is the probability of transitioning from node j to node i.
    """

    if is_sparse:
        M = sparse.csr_matrix(A, dtype=np.float64) 
    else:
        # make it numpy array, and make sure it is copied
        M = np.asarray(A, copy=True, dtype=np.float64)
    
    n = M.shape[0]
    outdeg = np.asarray( M.sum(axis=0) ).ravel()

    # non-dangling columns: normalize each column by its outdegree
    non_dangling = outdeg > 0
    M[:, non_dangling] = M[:, non_dangling] / outdeg[non_dangling]

    # dangling columns: uniform distribution
    dangling = outdeg == 0
    if np.any(dangling):
        M[:, dangling] = 1.0 / n

    return M

Finally, we have to actually reach the steady state to compute the final page rak.

In [4]:
def compute_pagerank(A, alpha=0.85, tol=1e-6, max_iter=100, is_sparse=False, verbose=False):
    """
    Parameters:
    - A: Adjacency matrix np.ndarray of shape (n, n)
    - alpha: Damping factor (float, default=0.85)
    - tol: Convergence tolerance (float, default=1e-6)
    - max_iter: Maximum number of iterations (int, default=100)
    - is_sparse: If True, the transition matrix will be treated as sparse (scipy.sparse.csr_matrix), otherwise dense (numpy array).
    - verbose: If True, prints convergence information at each iteration (bool, default=False)
    Returns:
    - pagerank: PageRank vector np.ndarray of shape (n,) where pagerank[i] is the PageRank score of node i.
    - iterations: Number of iterations taken to converge (int)
    - final_diff: Final L1 difference between the last two PageRank vectors (float)

    """
    
    M = build_transition_matrix(A, is_sparse)
    n = M.shape[0]

    # initial ranks
    pagerank = np.full(n, 1.0 / n, dtype=np.float64)

    for it in range(1, max_iter + 1):
        pagerank_next = alpha * (M @ pagerank) + (1.0 - alpha) * 1.0 / n
        pagerank_next = pagerank_next / pagerank_next.sum()  # normalize

        diff = np.abs(pagerank_next - pagerank).sum()  # L1
        if verbose:
            print(f"iter {it}: L1 diff = {diff:.3e}")

        if diff < tol:
            return pagerank_next, it, diff

        pagerank = pagerank_next

    return pagerank_next, max_iter, diff

# Improvements


The idea is that to achieve a better algorithmic efficiency, we can handle dead ends (nodes without outgoing edges) with a smarter technique. These nodes are handle with a random jump in the Markov process to any of the other nodes, wit probability $1/n$. Since it is deterministic, it is redundant to fullfill the matrix with this information, so we can save space and time by  handling dangling nodes differently.

In the standard formula, $$ pi^{t+1} = \beta M \pi^t + \frac{(1-\beta)}{n} $$. 
The term $\frac{(1-\beta)}{n}$ has to be added to every single cell of the resulting vector at each iteration. This means doing a lot of sums that could be computed separately and added just once: During adjency matrix $M$ multiplications (without dead ends) for $\pi^t$, the result won't summ up to 1 since the lack of dead ends contribution. Our solution will be structured this way:

1. identification of dead-end nodes
2. computation of page rank component accumulated in the dead ends, $S = \sum_{j \in \text{dead-ends}} \pi^t[j]$
3. computation of a single term contribution: $\text{Shared\_Contribution} = \frac{\beta \cdot S + (1 - \beta)}{n}$
4. the new page rank of each node will be: $\pi^{t+1}[i] = (\beta \cdot [M\pi^t]_i) + \text{Shared\_Contribution}$

To achieve this improbement, we'll need to modify both `compute pagerank` and `build_transition_matrix` functions.

In [12]:
def build_transition_matrix_enhanced(A):
    """
    we won't fill dangling columns with uniform distribution, but we will handle them during the PageRank iteration. 
    This way the transition matrix will remain sparse, saving memory and potentially improving performance for large graphs.

    parameters:
    - A: Adjacency matrix np.ndarray of shape (n, n)
    returns:
    - M: Transition matrix scipy.sparse.csr_matrix of shape (n, n) where M[i, j] is the probability of transitioning from node j to node i.
    - is_dangling: boolean array of shape (n,) where is_dangling[i] is True if node i is a dangling node (outdegree 0), False otherwise.
    """
    
    # creation of sparse matrix
    M = sparse.csr_matrix(A, dtype=np.float64)
    n = M.shape[0]

    # find outdegree
    outdeg = np.asarray(A.sum(axis=0)).ravel()
    
    # if outedeg is not zero, we normalize the column by its outdegree
    with np.errstate(divide='ignore'):
        inv_outdeg = 1.0 / outdeg
    inv_outdeg[np.isinf(inv_outdeg)] = 0  # set inf to 0 for dangling nodes


    # in order to normalize columns, we can multiply M by a diagonal matrix of inv_outdeg instead of dividing each column separately
    diag_mat = sparse.diags(inv_outdeg)
    M = M @ diag_mat

    is_dangling = (outdeg == 0)
    
    return M, is_dangling


the matrix so built won't sum up to 1 because we eliminated all the 1/n terms.

In [13]:
def compute_pagerank_enhanced(A, alpha=0.85, tol=1e-6, max_iter=100, verbose=False):
    """
    This version of PageRank handles dangling nodes during the iteration, without modifying the transition matrix. 
    This allows us to keep the transition matrix sparse, saving memory and potentially improving performance for large graphs.

    Parameters:
    - A: Adjacency matrix np.ndarray of shape (n, n)
    - alpha: Damping factor (float, default=0.85)
    - tol: Convergence tolerance (float, default=1e-6)
    - max_iter: Maximum number of iterations (int, default=100)
    - verbose: If True, prints convergence information at each iteration (bool, default=False)
    Returns:
    - pagerank: PageRank vector np.ndarray of shape (n,) where pagerank[i] is the PageRank score of node i.
    - iterations: Number of iterations taken to converge (int)
    - final_diff: Final L1 difference between the last two PageRank vectors (float)

    """
    
    M, is_dangling = build_transition_matrix_enhanced(A)
    n = M.shape[0]

    pagerank = np.full(n, 1.0 / n, dtype=np.float64) # initial uniform distribution

    for it in range(1, max_iter + 1):
        # contribution from non-dangling nodes
        pagerank_next = alpha * (M @ pagerank)

        # contribution from dangling nodes
        dangling_sum = np.sum(pagerank[is_dangling])
        dangling_contribution = alpha * dangling_sum / n
        pagerank_next += dangling_contribution

        # teleportation contribution
        pagerank_next += (1.0 - alpha) * 1.0 / n

        pagerank_next = pagerank_next / pagerank_next.sum()  # normalize

        diff = np.abs(pagerank_next - pagerank).sum()  # L1: diff between current and previous pagerank, used for convergence check (early stopping)
        if verbose:
            print(f"iter {it}: L1 diff = {diff:.3e}")

        if diff < tol:
            return pagerank_next, it, diff

        pagerank = pagerank_next

    return pagerank_next, max_iter, diff    

# Benchmarks

Now, Let's see how it performs over the Marvel Dataset. We need a function to test the performance many times in order to lower the variance of the results.
We need an auxiliary functions to trasnform the raw data to a useful typed list to build the adjency matrix

In [5]:
MARVEL_RAW_EDGES = "hero-network.csv"

In [6]:

def read_edges_with_counts_marvel_csv(filename, has_header=True):
    """ 
    Parameters:
    - filename: Path to the CSV file containing edges (str)
    - has_header: If True, the first row of the CSV file will be treated as a header and skipped (bool, default=True)
    Returns:
    - edges: List of tuples (source, target, count) where source and target are node names (str) and count is the number of edges between them (int)
    """
    edge_counts = {}

    with open(filename, newline='', encoding="utf-8") as f:
        reader = csv.reader(f)

        if has_header:
            next(reader, None)  # skip header

        for row in reader:
            source = row[0].strip()
            target = row[1].strip()

            if source < target:
                key = (source, target)
            else:
                key = (target, source)

            if key in edge_counts:
                edge_counts[key] += 1
            else:
                edge_counts[key] = 1

    return [(src, tgt, count) for (src, tgt), count in edge_counts.items()]


In [7]:
edges = read_edges_with_counts_marvel_csv(MARVEL_RAW_EDGES)
A, names = build_adjacency_Matrix(edges) 

### Tests

In [26]:
def run_tests(A, method, repetitions):
    print("\n" + "="*40)
    print(f"Running tests for method {method}")
    print( "="*40 + "\n")
    # cold staart
    match method:
        case 1:
            compute_pagerank(A, alpha=0.85, tol=1e-9, max_iter=200, is_sparse=False, verbose=False)
        case 2:
            compute_pagerank(A, alpha=0.85, tol=1e-9, max_iter=200, is_sparse=True, verbose=False)
        case 3: 
            compute_pagerank_enhanced(A, alpha=0.85, tol=1e-9, max_iter=200, verbose=False)

    # benchmarks
    start = time.perf_counter()

    for _ in range(repetitions):
        match method:
            case 1:
                r, iters, d = compute_pagerank(A, alpha=0.85, tol=1e-9, max_iter=200, is_sparse=False, verbose=False)
            case 2:
                r, iters, d = compute_pagerank(A, alpha=0.85, tol=1e-9, max_iter=200, is_sparse=True, verbose=False)
            case _: 
                r, iters, d = compute_pagerank_enhanced(A, alpha=0.85, tol=1e-9, max_iter=200, verbose=False)
        

    elapsed = time.perf_counter() - start
    match method:
        case 1:
            footprint = asizeof.asizeof( build_transition_matrix(A, is_sparse=False) )
        case 2:
            footprint = asizeof.asizeof( build_transition_matrix(A, is_sparse=True) )
        case _:
            footprint = asizeof.asizeof( build_transition_matrix_enhanced(A) )


    print (f"Elapsed time (avg): {(elapsed/repetitions):.2f} sec.s")
    print(f"Memory footprint {footprint/10**6:.2f} MB.")

    print("PageRank:", r[:4], "...")
    print("iters:", iters)
    print("sum:", r.sum())

In [27]:
run_tests(A, method=1, repetitions=10)
run_tests(A, method=2, repetitions=10)
run_tests(A, method=3, repetitions=10)


Running tests for method 1

Elapsed time (avg): 5.29 sec.s
Memory footprint 329.83 MB.
PageRank: [6.29994719e-05 2.85812118e-04 1.81257318e-04 8.82790875e-05] ...
iters: 64
sum: 1.0000000000000002

Running tests for method 2

Elapsed time (avg): 3.26 sec.s
Memory footprint 8.05 MB.
PageRank: [6.29994719e-05 2.85812118e-04 1.81257318e-04 8.82790875e-05] ...
iters: 64
sum: 1.0

Running tests for method 3

Elapsed time (avg): 0.32 sec.s
Memory footprint 8.05 MB.
PageRank: [6.29994719e-05 2.85812118e-04 1.81257318e-04 8.82790875e-05] ...
iters: 64
sum: 1.0
